# 0073: the reader's read on dev (arms M, W and P)

The answers only. Each instruction of dev (300) is read whole, its answer held token by token to the grammar of its
own instruction. Five passes:

- **arm M** and **arm W**, each twice: free, and with the rule that a description may only use the numbers, strings
  and views its line's own text gives (point 7 of the order). Dev decides which variant each arm keeps.
- **arm P**: round 2's adapter (`cubby450m_view_reader_v2`), with its own form and grammar: the baseline.

Dev is for looking; test is not read here. The answers are far shorter than round 2's programs, so the four passes
of M and W are quick; arm P's pass is round 2's read (about ten minutes for 300 rows on an A100).

It reads, from Drive: the base, the tokenizer, the three adapters and `rules/long_rows_0073/read_rows.jsonl`; it
writes `rules/reads/0073/*_dev.jsonl` (resumable: a rerun keeps the rows already answered).
**Disconnect when it prints `read: dev`.**

In [ ]:
import hashlib, json, os, subprocess, sys
BRANCH, PHASE = 'cowork/0073', 'dev'
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q -b {BRANCH} https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git fetch -q && git checkout -q {BRANCH} && git pull -q --ff-only
!pip -q install tokenizers safetensors numpy regex
from google.colab import drive; drive.mount('/content/drive')
REPO, DRIVE = '/content/CubbyLLM', '/content/drive/MyDrive/cubbyllm'
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'
TOKENIZER = f'{DRIVE}/token_cache_cont/tokenizer/grillcheese_bbpe128k.json'
ROWS      = f'{DRIVE}/rules/long_rows_0073/read_rows.jsonl'
ADAPTERS  = {'m': f'{DRIVE}/rules/cubby450m_view_split_m', 'w': f'{DRIVE}/rules/cubby450m_view_split_w',
             'p': f'{DRIVE}/rules/cubby450m_view_reader_v2'}
READS     = f'{DRIVE}/rules/reads/0073'
for p in [CKPT, TOKENIZER, ROWS] + [f'{d}/talk_lora.{x}' for d in ADAPTERS.values() for x in ('safetensors', 'json')]:
    assert os.path.exists(p), p
for arm in 'mw':
    assert os.path.exists(f"{ADAPTERS[arm]}/DONE"), f'arm {arm} is not trained to the end'
man = json.load(open(f'{REPO}/validation/logs/long_rows_0073_read_rows.manifest.json'))
assert hashlib.sha256(open(ROWS, 'rb').read()).hexdigest() == man['sha256'], 'read_rows.jsonl is not the committed file'
keys = {k for ln in open(ROWS, encoding='utf-8') for k in json.loads(ln)}
assert keys == {'id', 'split', 'instruction'}, f'the rows file holds more than ids and instructions: {keys}'
os.makedirs(READS, exist_ok=True)
print(man['rows'], 'rows', man['by_split'])
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SPLITS, N = 'dev', 300
PASSES = [('m', False), ('m', True), ('w', False), ('w', True), ('p', False)]     # (arm, the own-text rule)
def out_of(arm, own):
    return f"{READS}/view_reader_v2_dev.jsonl" if arm == 'p' else f"{READS}/view_split_{arm}_{'own' if own else 'free'}_dev.jsonl"

os.environ['CB_NO_FLEX'] = '1'
want = sum(min(N, man['by_split'][s]) for s in SPLITS.split(','))
for arm, own in PASSES:
    OUT = out_of(arm, own)
    have = [json.loads(ln) for ln in open(OUT, encoding='utf-8')] if os.path.exists(OUT) else []
    assert all(bool(d.get('own_text')) == own for d in have), f'{OUT} was read in another variant: move it aside'
    if len({d['id'] for d in have}) >= want:
        print(f'--- arm {arm}: {OUT} is whole ({len(have)} answers): not read again')
        continue
    print(f"--- arm {arm}{' (own text)' if own else ''}: {SPLITS} -> {OUT}")
    cmd = [sys.executable, '-u', 'validation/exp_view_split_torch.py', '--arm', arm, '--ckpt', CKPT, '--tokenizer', TOKENIZER,
           '--adapter', ADAPTERS[arm], '--rows', ROWS, '--splits', SPLITS, '--n', str(N), '--batch', '32', '--out', OUT]
    p = subprocess.Popen(cmd + (['--own-text'] if own else []), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'the read failed ({p.returncode})'
    !wc -l "{OUT}"
print('read:', PHASE)

## Done

**Disconnect the runtime.** At home the answers are scored on the VM (`validation/exp_view_split_score.py`; arm P
with round 2's scorer), dev's counts decide point 7 for each arm, and that choice is committed. Only then does
`view_split_read_test.ipynb` read test and test_long.